# 04 — Review: Load and Summarize Model Outputs

## Mini-Project: Full Pipeline

No new concepts in this notebook — this is all practice.

You'll build a complete analysis pipeline, end to end:

**load → inspect → summarize → save**

This is the workflow you'll run constantly in AI safety research:
load a file of model outputs, compute statistics, identify patterns, save results.

Work through each step. Check cells will verify your answers as you go.

## Setup

In [ ]:
import json
import csv
import os
import sys
import pandas as pd
from pathlib import Path

sys.path.insert(0, "../../")
from src.checks import check_equal, check_type, check_contains, check_length, check_keys

# Find the repo root by walking up until we find pyproject.toml
def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    raise FileNotFoundError("Could not find repo root")

_REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(_REPO_ROOT))
from src.checks import check_equal, check_type, check_contains, check_length, check_keys

JSON_PATH = _REPO_ROOT / "data" / "synthetic" / "model_outputs.json"
CSV_PATH  = _REPO_ROOT / "data" / "synthetic" / "evaluation_results.csv"

print("Setup complete")
print("JSON_PATH exists:", JSON_PATH.exists())
print("CSV_PATH exists: ", CSV_PATH.exists())

os.makedirs("../../data/synthetic", exist_ok=True)
print("Setup complete")

## Step 1: Load the model outputs

Load `model_outputs.json` using `json.load()` and store the result in `outputs`.

In [ ]:
# YOUR CODE HERE
outputs = None
print(f"Outputs: {outputs}")

In [ ]:
check_type(outputs, list, "outputs is a list")
check_length(outputs, 20, "outputs has 20 records")
# Each record should have these keys
check_keys(outputs[0], ["id", "model", "prompt", "response", "flagged", "category"], "record has expected keys")

## Step 2: Total number of outputs

Store the total count of records in `total_outputs`.

In [ ]:
# YOUR CODE HERE
total_outputs = len(outputs)
print(f"Total outputs: {total_outputs}")

In [ ]:
check_equal(total_outputs, 20, "total_outputs == 20")

## Step 3: All unique models

Find all unique model names in the dataset. Store them as a **sorted list** in `unique_models`.

Hint: use a set to deduplicate, then `sorted()` to get a consistent order.

In [ ]:
# YOUR CODE HERE
unique_models = []
print(f"Models: {unique_models}")

In [ ]:
check_equal(unique_models, ["model-a-v1", "model-b-v1"], "unique models, sorted")
check_length(unique_models, 2, "exactly 2 unique models")

## Step 4: Count flagged vs. non-flagged outputs

Count how many outputs are flagged and how many are not.
Store the counts in `flagged_count` and `not_flagged_count`.

In [ ]:
# YOUR CODE HERE
flagged_count = 0
not_flagged_count = 0
print(f"Flagged count: {flagged_count}")
print(f"Not flagged count: {not_flagged_count}")

In [ ]:
check_equal(flagged_count, 7, "7 flagged outputs")
check_equal(not_flagged_count, 13, "13 non-flagged outputs")
check_equal(flagged_count + not_flagged_count, 20, "counts sum to 20")

## Step 5: Count by category

Build a dict mapping each `category` value to its count. Include `None` for unflagged outputs.
Store it in `category_counts`.

Hint: use a dict and loop through `outputs`, incrementing `counts[record["category"]]`.
Or use `dict.get(key, 0)` to safely increment missing keys.

In [ ]:
# YOUR CODE HERE
category_counts = None

print(category_counts)


In [ ]:
check_type(category_counts, dict, "category_counts is a dict")
check_equal(category_counts[None], 13, "13 outputs with category None")
check_equal(category_counts["misinformation"], 3, "3 misinformation outputs")
check_equal(category_counts["harmful_content"], 1, "1 harmful_content output")

## Step 6: Per-model flagging rate

Compute the flagging rate (flagged / total) for each model.
Store the result in `flagging_rates` — a dict mapping model name to a float between 0 and 1.

Example: `{"model-a-v1": 0.0, "model-b-v1": 0.777...}`

In [ ]:
# YOUR CODE HERE
flagging_rates = {}

print("Flagging rates:")
if flagging_rates:
    for model, rate in flagging_rates.items():
        print(f"  {model}: {rate:.1%}")

In [ ]:
check_type(flagging_rates, dict, "flagging_rates is a dict")
check_keys(flagging_rates, ["model-a-v1", "model-b-v1"], "flagging_rates has both models")
check_equal(flagging_rates["model-a-v1"], 0.0, "model-a-v1 has 0% flagging rate")
check_equal(round(flagging_rates["model-b-v1"], 4), round(7/9, 4), "model-b-v1 flagging rate is 7/9")

## Step 7: Load the evaluation results CSV with pandas

Load `evaluation_results.csv` using `pd.read_csv()`. Store the DataFrame in `eval_df`.
Print its shape and first few rows.

In [ ]:
# YOUR CODE HERE
eval_df = None

if eval_df is not None:
    print("Shape:", eval_df.shape)
    print(eval_df.head())

In [ ]:
check_type(eval_df, pd.DataFrame, "eval_df is a DataFrame")
check_equal(eval_df.shape[0], 20, "DataFrame has 20 rows")
check_equal(list(eval_df.columns[:3]), ["model", "task", "score"], "first 3 columns are model, task, score")

## Step 8: Find the model+task combination with the highest score

Find the row in `eval_df` with the maximum score.
Store:
- the model name in `best_model`
- the task name in `best_task`
- the score in `best_score`

Hint: `df.loc[df["score"].idxmax()]` returns the row with the highest score.

In [ ]:
# YOUR CODE HERE
best_model = None
best_task = None
best_score = None
print(f"Best: {best_model} on {best_task} with score {best_score}")

In [ ]:
check_equal(best_model, "model-a-v2", "best model is model-a-v2")
check_equal(best_task, "harmful_refusal", "best task is harmful_refusal")
check_equal(best_score, 0.98, "best score is 0.98")

## Step 9: Save a summary as JSON

Build a summary dict and save it to `"../../data/synthetic/summary.json"` with `indent=2`.

The summary dict should have these keys:
- `"total_outputs"` — int
- `"unique_models"` — list of model names
- `"flagged_count"` — int
- `"not_flagged_count"` — int
- `"flagging_rates"` — dict of model → float
- `"category_counts"` — dict of category → int (convert None key to the string `"none"`)
- `"best_model"` — string
- `"best_task"` — string
- `"best_score"` — float

Note: JSON keys must be strings — convert the `None` key in `category_counts` to `"none"`.

In [ ]:
# Build the summary dict — replace None key with "none" for JSON compatibility
summary = None

if summary:
    print("Summary saved. Preview:")
    print(json.dumps(summary, indent=2))

In [ ]:
check_type(summary, dict, "summary is a dict")
check_keys(
    summary,
    ["total_outputs", "unique_models", "flagged_count", "not_flagged_count",
     "flagging_rates", "category_counts", "best_model", "best_task", "best_score"],
    "summary has all required keys"
)
check_equal(summary["total_outputs"], 20, "summary total_outputs")
check_equal(summary["best_model"], "model-a-v2", "summary best_model")

# Verify the file was actually written
import os
assert os.path.exists("../../data/synthetic/summary.json"), "summary.json was not created!"
print("summary.json exists on disk.")

## Reflection

Congratulations — you just built a complete research analysis pipeline.

Look at what you did:

1. **Loaded** a JSON file of model outputs from disk
2. **Inspected** its structure and counted records
3. **Summarized** it — unique models, flagging counts, per-category breakdown
4. **Computed statistics** — per-model flagging rates (a safety signal!)
5. **Loaded a CSV** with evaluation scores
6. **Found the best performer** across all model/task combinations
7. **Saved a summary** back to disk for sharing and reproducibility

This is the workflow you'll run constantly in AI safety research:
load a file of model outputs, compute statistics, identify patterns, save results.

In real work, the files are larger, the categories are more complex, and the statistics are
more sophisticated — but the structure is exactly this. You now have the foundation.

## Module 02 Complete!

You've covered:

| Notebook | Topic |
|---|---|
| 01 | Imports and modules — `math`, `os`, `sys`, `pathlib`, `random` |
| 02 | Files and I/O — `open()`, `with`, reading/writing text files |
| 03 | JSON and CSV — parsing, loading, summarizing structured data |
| 04 | Review — full load → inspect → summarize → save pipeline |

**Next up: Module 03** — Writing Python scripts and working with the command line.